# Sub-Category: PGA-UNet vs U-Net vs Attention U-Net

| Section | Nội dung |
|---|---|
| **0+1** | U-Net chạy **toàn bộ** → sort Dice → **easy_stems (top-50)** + **hard_stems (bot-50)** — dùng chung cho cả 3 model; **metrics N=50**, **visualize top-10** |
| **2** | Att-UNet: metrics N=50, visualize top-10 |
| **3** | PGA-UNet: metrics N=50 stems (số poly ≥ 50), visualize polygon từ **10 stems đầu** |
| **Agg** | Bảng tổng hợp 3 model × Easy / Hard (N=50) + bar chart |

Visualization 4 cột: **Ảnh + Prompt BBox** | **GT** | **Dự đoán** *(Dice/IoU/CBL)* | **Overlap** *(Pre/Rec/HD95)*

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1 — SETUP (clone, dataset, checkpoints, utilities)
# ══════════════════════════════════════════════════════
%cd /content
import os, sys, cv2, json as _json
import numpy as np, torch
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle as Rect
!pip install -q gdown tqdm opencv-python matplotlib scipy
import gdown

REPO='https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git'
for d,b in [('unet-repo',None),('attunet-repo','attention-unet2D'),('pga-repo','TN_B_ON')]:
    if not os.path.exists(f'/content/{d}'):
        br=f'-b {b} ' if b else ''
        os.system(f'git clone -q {br}{REPO} /content/{d}')
    print(f'  ✅ {d}')

if not os.path.exists('/content/dataset_BTXRD'):
    gdown.download('https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW',
                   '/content/dataset_BTXRD.zip', quiet=False)
    os.system('unzip -q /content/dataset_BTXRD.zip -d /content/')
for d in ['unet-repo','attunet-repo','pga-repo']:
    if not os.path.exists(f'/content/{d}/dataset_BTXRD'):
        os.system(f'cp -r /content/dataset_BTXRD /content/{d}/dataset_BTXRD')
print('  ✅ Dataset ready')

os.makedirs('/content/checkpoints',exist_ok=True)
for cid,fn in [('1hKyyw8WvN6WRyzjDE50upWI0NIDJbSMu','unet_best.pth'),
               ('1ONFzzZS4vLg6P3FN887h4v62cBymZCGb','attunet_best.pth'),
               ('1sSoQ8SObteWETuKSsGGhBASPtXQz9JbY', 'pga_unet_expB_best.pth')]:
    p=f'/content/checkpoints/{fn}'
    if not os.path.exists(p): gdown.download(f'https://drive.google.com/uc?id={cid}',p,quiet=False)
    print(f'  ✅ {fn}  {os.path.getsize(p)//1024}KB')

DEVICE,IMG_SIZE,ZOOM_R=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),512,0.30
IMG_DIR ='/content/unet-repo/dataset_BTXRD/test/images'
MASK_DIR='/content/unet-repo/dataset_BTXRD/test/masks'
JSON_DIR='/content/pga-repo/dataset_BTXRD/test/annotations'
KEYS=['dice','iou','precision','recall','hd95','cbl']

def extract_lcc(m):
    if m.sum()==0: return m
    n,lbl,st,_=cv2.connectedComponentsWithStats(m.astype(np.uint8),8)
    return m if n<=1 else (lbl==(1+np.argmax(st[1:,cv2.CC_STAT_AREA]))).astype(np.float32)

def calc_metrics(prob,gt,eps=1e-6):
    pm=extract_lcc((prob>0.5).astype(np.float32)); gm=(gt>0.5).astype(np.float32)
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    p,g=pm.astype(bool),gm.astype(bool); hd95=float(IMG_SIZE)
    if p.any() and g.any():
        pe=p^binary_erosion(p); ge=g^binary_erosion(g)
        d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
        if len(d1) and len(d2): hd95=float(max(np.percentile(d1,95),np.percentile(d2,95)))
    if gm.sum()==0 or pm.sum()==0: cbl=0.
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)),iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95,cbl=cbl,mask=pm)

def plateau_heatmap(bbox,S=512):
    x1=max(0,int(bbox[0])-5);y1=max(0,int(bbox[1])-5)
    x2=min(S,int(bbox[2])+5);y2=min(S,int(bbox[3])+5)
    hm=np.zeros((S,S),dtype=np.float32)
    if x2>x1 and y2>y1: hm[y1:y2,x1:x2]=1.0; hm=cv2.GaussianBlur(hm,(31,31),0)
    return hm

def print_metrics(label,mlist):
    m={k:np.mean([r[k] for r in mlist]) for k in KEYS}
    print(f"  {label:<20} Dice={m['dice']:.4f} IoU={m['iou']:.4f} Pre={m['precision']:.4f}"
          f" Rec={m['recall']:.4f} HD95={m['hd95']:.2f} CBL={m['cbl']:.4f}")
    return m

def visualize(records, title, n=15, show_bbox=False):
    recs=records[:n]; nr=len(recs)
    fig,axes=plt.subplots(nr,4,figsize=(20,4.5*nr),squeeze=False)
    fig.suptitle(title,fontsize=13,fontweight='bold',y=1.005)
    for j,t in enumerate(['Ảnh gốc','Ground Truth',
                           'Dự đoán\n(Dice / IoU / CBL)',
                           'Overlap\n(Pre / Rec / HD95)']):
        axes[0,j].set_title(t,fontsize=9,fontweight='bold')
    for row,rec in enumerate(recs):
        # UNet dataset normalize [0,1] → dùng *255 để hiển thị
        gray=np.clip(rec['img_np']*255,0,255).astype(np.uint8)
        rgb=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)
        gt=rec['gt']; pm=rec['m']['mask']
        def ov(mask,clr,a=0.5):
            o=rgb.copy().astype(np.float32); o[mask>0.5]=o[mask>0.5]*(1-a)+np.array(clr)*a
            return np.clip(o,0,255).astype(np.uint8)
        diff=rgb.copy(); pb,gb=pm>0.5,gt>0.5
        diff[gb&~pb]=[30,180,30]; diff[pb&~gb]=[220,60,60]; diff[pb&gb]=[220,200,0]
        ax=axes[row,0]; ax.imshow(gray,cmap='gray')
        if show_bbox:
            for bx in rec.get('bboxes',[]):
                ax.add_patch(Rect((bx[0],bx[1]),bx[2]-bx[0],bx[3]-bx[1],lw=1.5,edgecolor='lime',facecolor='none'))
        ax.set_ylabel(rec['stem'],fontsize=7,rotation=0,labelpad=75,va='center'); ax.axis('off')
        axes[row,1].imshow(ov(gt,[30,120,255])); axes[row,1].axis('off')
        m=rec['m']
        axes[row,2].imshow(ov(pm,[220,60,60]))
        axes[row,2].set_title(f"Dice={m['dice']:.3f}  IoU={m['iou']:.3f}  CBL={m['cbl']:.3f}",
                              fontsize=8,color='darkred',pad=2); axes[row,2].axis('off')
        axes[row,3].imshow(diff)
        axes[row,3].set_title(f"Pre={m['precision']:.3f}  Rec={m['recall']:.3f}  HD95={m['hd95']:.1f}px",
                              fontsize=8,color='saddlebrown',pad=2); axes[row,3].axis('off')
    plt.tight_layout()
    fname=title.split('|')[0].strip()[:6].lower()+'_'+title.split('|')[-1].strip()[:4].lower()+'.png'
    plt.savefig(fname,dpi=100,bbox_inches='tight'); plt.show(); print(f'  ✅ {fname}')

# Load dataset & image stems
sys.path.insert(0,'/content/unet-repo')
from dataset import BTXRD_Dataset
test_ds=BTXRD_Dataset(image_dir=IMG_DIR,mask_dir=MASK_DIR,img_size=IMG_SIZE,is_train=False)
# UNet dataset dùng os.listdir() không sort → phải sort lại để khớp với img_stems
test_ds.images=sorted(test_ds.images)

img_stems=[os.path.splitext(f)[0] for f in test_ds.images]  # lấy stem từ đúng thứ tự dataset

all_raw=[]
for idx,(img_t,mask_t) in enumerate(DataLoader(test_ds,batch_size=1,shuffle=False)):
    all_raw.append({'idx':idx,
                    'img_np':img_t[0,0].numpy().copy(),   # [0,1]
                    'gt':mask_t[0,0].numpy().copy(),
                    'img_t':img_t.clone(),
                    'stem':img_stems[idx]})
SEC={}
print(f'\n✅ Setup xong | {len(all_raw)} samples | device={DEVICE}')
print(f'   Ví dụ stems: {img_stems[:3]} ...')


In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 0+1 — U-Net 2D
# Chạy toàn bộ → sort Dice → easy/hard stems
# Metrics: N_METRIC=50 | Visualize: N_SHOW=10
# ══════════════════════════════════════════════════════
N_METRIC=50   # số ảnh dùng để tính độ đo
N_SHOW  =10   # số ảnh hiển thị

%cd /content/unet-repo
for _k in list(sys.modules.keys()):
    if 'models' in _k: del sys.modules[_k]
if '/content/unet-repo' not in sys.path:
    sys.path.insert(0, '/content/unet-repo')
else:
    sys.path.remove('/content/unet-repo'); sys.path.insert(0, '/content/unet-repo')

from models.networks.unet_2D import unet_2D
unet=unet_2D(in_channels=1,n_classes=1).to(DEVICE)
unet.load_state_dict(torch.load('/content/checkpoints/unet_best.pth',map_location=DEVICE,weights_only=True))
unet.eval(); print('✅ U-Net loaded')

all_unet_res=[]
with torch.no_grad():
    for s in tqdm(all_raw,'UNet-All'):
        prob=torch.sigmoid(unet(s['img_t'].to(DEVICE)))[0,0].cpu().numpy()
        all_unet_res.append(calc_metrics(prob,s['gt']))
del unet

sorted_all=sorted(range(len(all_raw)),key=lambda i:all_unet_res[i]['dice'],reverse=True)
easy_idx=sorted_all[:N_METRIC]
hard_idx=sorted_all[-N_METRIC:][::-1]

easy_stems=[all_raw[i]['stem'] for i in easy_idx]
hard_stems =[all_raw[i]['stem'] for i in hard_idx]

print(f'\n  Easy top-{N_METRIC}  Dice: {all_unet_res[easy_idx[0]]["dice"]:.4f} → {all_unet_res[easy_idx[-1]]["dice"]:.4f}')
print(f'  Hard bot-{N_METRIC}  Dice: {all_unet_res[hard_idx[0]]["dice"]:.4f} → {all_unet_res[hard_idx[-1]]["dice"]:.4f}')
print(f'\neasy_stems = {easy_stems}')
print(f'hard_stems  = {hard_stems}')

bar='='*70
print(f'\n{bar}\n  SECTION 1 — U-Net  (metrics N={N_METRIC}, show N={N_SHOW})\n{bar}')
SEC['unet']={
    'easy': print_metrics(f'Easy (N={N_METRIC})', [all_unet_res[i] for i in easy_idx]),
    'hard': print_metrics(f'Hard (N={N_METRIC})', [all_unet_res[i] for i in hard_idx])
}

stem2idx={all_raw[i]['stem']:i for i in range(len(all_raw))}
recs_e=[dict(**all_raw[stem2idx[st]], m=all_unet_res[stem2idx[st]]) for st in easy_stems]
recs_h=[dict(**all_raw[stem2idx[st]], m=all_unet_res[stem2idx[st]]) for st in hard_stems]
visualize(recs_e, f'U-Net | EASY — top-{N_SHOW} / {N_METRIC} ảnh Dice cao nhất', n=N_SHOW)
visualize(recs_h, f'U-Net | HARD — bottom-{N_SHOW} / {N_METRIC} ảnh Dice thấp nhất', n=N_SHOW)


In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 2 — Attention U-Net 2D
# Metrics: N_METRIC=50 | Visualize: N_SHOW=10
# ══════════════════════════════════════════════════════

%cd /content/attunet-repo
for _k in list(sys.modules.keys()):
    if 'models' in _k: del sys.modules[_k]
if '/content/attunet-repo' not in sys.path:
    sys.path.insert(0, '/content/attunet-repo')
else:
    sys.path.remove('/content/attunet-repo'); sys.path.insert(0, '/content/attunet-repo')

from models.networks.attention_unet_2D import Attention_UNet_2D
attunet=Attention_UNet_2D(in_channels=1,n_classes=1).to(DEVICE)
attunet.load_state_dict(torch.load('/content/checkpoints/attunet_best.pth',map_location=DEVICE,weights_only=True))
attunet.eval(); print('✅ Att-UNet loaded')

res_a_easy,res_a_hard=[],[]
with torch.no_grad():
    for st in tqdm(easy_stems,'AttUNet-Easy'):
        i=stem2idx[st]
        prob=torch.sigmoid(attunet(all_raw[i]['img_t'].to(DEVICE)))[0,0].cpu().numpy()
        res_a_easy.append(calc_metrics(prob,all_raw[i]['gt']))
    for st in tqdm(hard_stems,'AttUNet-Hard'):
        i=stem2idx[st]
        prob=torch.sigmoid(attunet(all_raw[i]['img_t'].to(DEVICE)))[0,0].cpu().numpy()
        res_a_hard.append(calc_metrics(prob,all_raw[i]['gt']))
del attunet

bar='='*70
print(f'\n{bar}\n  SECTION 2 — Att-UNet  (metrics N={N_METRIC}, show N={N_SHOW})\n{bar}')
SEC['attunet']={
    'easy': print_metrics(f'Easy (N={N_METRIC})', res_a_easy),
    'hard': print_metrics(f'Hard (N={N_METRIC})', res_a_hard)
}

recs_e=[dict(**all_raw[stem2idx[st]], m=res_a_easy[k]) for k,st in enumerate(easy_stems)]
recs_h=[dict(**all_raw[stem2idx[st]], m=res_a_hard[k]) for k,st in enumerate(hard_stems)]
visualize(recs_e, f'Att-UNet | EASY — top-{N_SHOW} / {N_METRIC} ảnh', n=N_SHOW)
visualize(recs_h, f'Att-UNet | HARD — bottom-{N_SHOW} / {N_METRIC} ảnh', n=N_SHOW)


In [ ]:
# ══════════════════════════════════════════════════════
# SECTION 3 — PGA-UNet
# Metrics: tất cả polygon từ N_METRIC=50 stems
# Visualize: polygon từ N_SHOW=10 stems đầu (có thể >10 hàng nếu multi-polygon)
# Chạy độc lập: paste easy_stems/hard_stems từ output cell-2 vào _EASY/_HARD
# ══════════════════════════════════════════════════════

# ── Chỉ điền khi chạy ĐỘC LẬP ──────────────────────
_EASY = []   # paste: ['img001', ...]
_HARD  = []  # paste: ['img050', ...]
_N_METRIC = 50
_N_SHOW   = 10
# ────────────────────────────────────────────────────

if 'easy_stems' not in dir() or not easy_stems:
    easy_stems, hard_stems = _EASY, _HARD
if 'N_METRIC' not in dir(): N_METRIC=_N_METRIC
if 'N_SHOW'   not in dir(): N_SHOW  =_N_SHOW
assert easy_stems and hard_stems, "❌ Paste easy_stems/hard_stems từ output cell-2 vào _EASY/_HARD!"

%cd /content/pga-repo
for _k in list(sys.modules.keys()):
    if 'models' in _k: del sys.modules[_k]
if '/content/pga-repo' not in sys.path:
    sys.path.insert(0, '/content/pga-repo')
else:
    sys.path.remove('/content/pga-repo'); sys.path.insert(0, '/content/pga-repo')

from models.networks.prompt_unet_2D import PGA_UNet
pga=PGA_UNet(in_channels=1,n_classes=1,use_encoder_prompt=True).to(DEVICE)
pga.load_state_dict(torch.load('/content/checkpoints/pga_unet_expB_best.pth',
                                map_location=DEVICE,weights_only=True))
pga.eval(); print('✅ PGA-UNet loaded')

import importlib.util
_spec=importlib.util.spec_from_file_location('pga_ds_mod','/content/pga-repo/dataset.py')
_mod=importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_mod)
PGA_Dataset=_mod.BTXRD_Dataset

pga_ds=PGA_Dataset(image_dir=IMG_DIR, json_dir=JSON_DIR,
                   img_size=IMG_SIZE, is_train=False, prompt_mode='zoom_out')
print(f'  PGA dataset: {len(pga_ds)} total polygon-samples')

stem_to_pga={}
for i,(img_name,_) in enumerate(pga_ds.all_samples):
    st=os.path.splitext(img_name)[0]
    stem_to_pga.setdefault(st,[]).append(i)

def mask_to_bbox(mask):
    ys,xs=np.where(mask>0.5)
    if not len(xs): return None
    bw=float(xs.max()-xs.min()); bh=float(ys.max()-ys.min())
    return [max(0.,float(xs.min())-bw*ZOOM_R), max(0.,float(ys.min())-bh*ZOOM_R),
            min(float(IMG_SIZE),float(xs.max())+bw*ZOOM_R),
            min(float(IMG_SIZE),float(ys.max())+bh*ZOOM_R)]

def build_pga_from_dataset(stems):
    """Mỗi polygon trong JSON = 1 sample. img/gt/hm_t đều từ cùng pga_ds."""
    samples=[]
    for stem in stems:
        pga_idxs=stem_to_pga.get(stem,[])
        if not pga_idxs:
            print(f'  ⚠ {stem} không có JSON/polygon'); continue
        for k,pga_idx in enumerate(pga_idxs):
            img_t,gt_t,hm_t=pga_ds[pga_idx]
            gt_np=gt_t[0].numpy()
            bbox=mask_to_bbox(gt_np)
            if bbox is None: continue
            samples.append(dict(stem=stem, poly_idx=k,
                                img_np=img_t[0].numpy(),
                                img_t=img_t.unsqueeze(0),
                                gt=gt_np,
                                hm_t=hm_t.unsqueeze(0),
                                bbox=bbox))
    return samples

# Build từ toàn bộ N_METRIC stems
easy_pga=build_pga_from_dataset(easy_stems)
hard_pga =build_pga_from_dataset(hard_stems)
print(f'  Easy: {len(easy_pga)} poly từ {len(easy_stems)} ảnh')
print(f'  Hard: {len(hard_pga)} poly từ {len(hard_stems)} ảnh')

with torch.no_grad():
    for s in tqdm(easy_pga+hard_pga,'PGA-Inference'):
        prob=torch.sigmoid(pga(s['img_t'].to(DEVICE), s['hm_t'].to(DEVICE)))[0,0].cpu().numpy()
        s['m']=calc_metrics(prob,s['gt'])
del pga

bar='='*70
print(f'\n{bar}\n  SECTION 3 — PGA-UNet  (metrics N={N_METRIC} stems, show N={N_SHOW} stems)\n{bar}')
SEC['pga']={
    'easy': print_metrics(f"Easy ({len(easy_pga)} poly / {len(easy_stems)} ảnh)",[s['m'] for s in easy_pga]),
    'hard': print_metrics(f"Hard ({len(hard_pga)} poly / {len(hard_stems)} ảnh)",[s['m'] for s in hard_pga]),
    'n_easy': len(easy_pga),
    'n_hard': len(hard_pga),
}

# Visualize: chỉ lấy polygon từ N_SHOW stems đầu
show_easy_set=set(easy_stems[:N_SHOW])
show_hard_set =set(hard_stems[:N_SHOW])
easy_pga_vis=[s for s in easy_pga if s['stem'] in show_easy_set]
hard_pga_vis =[s for s in hard_pga  if s['stem'] in show_hard_set]

def pga_vis_all(samples, title):
    nr=len(samples)
    if nr==0: print('  (no samples)'); return
    fig,axes=plt.subplots(nr,4,figsize=(20,4.5*nr),squeeze=False)
    fig.suptitle(title,fontsize=13,fontweight='bold',y=1.005)
    for j,t in enumerate(['Ảnh gốc + Prompt BBox','Ground Truth',
                           'Dự đoán\n(Dice / IoU / CBL)',
                           'Overlap\n(Pre / Rec / HD95)']):
        axes[0,j].set_title(t,fontsize=9,fontweight='bold')
    for row,s in enumerate(samples):
        gray=np.clip((s['img_np']*0.5+0.5)*255,0,255).astype(np.uint8)
        rgb=cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)
        gt=s['gt']; pm_mask=s['m']['mask']; bx=s['bbox']
        def ov(mask,clr,a=0.5):
            o=rgb.copy().astype(np.float32)
            o[mask>0.5]=o[mask>0.5]*(1-a)+np.array(clr)*a
            return np.clip(o,0,255).astype(np.uint8)
        diff=rgb.copy(); pb,gb=pm_mask>0.5,gt>0.5
        diff[gb&~pb]=[30,180,30]; diff[pb&~gb]=[220,60,60]; diff[pb&gb]=[220,200,0]
        ax=axes[row,0]; ax.imshow(gray,cmap='gray')
        ax.add_patch(Rect((bx[0],bx[1]),bx[2]-bx[0],bx[3]-bx[1],
                          lw=1.5,edgecolor='lime',facecolor='none'))
        lbl=s['stem'] if s['poly_idx']==0 else f"{s['stem']} #{s['poly_idx']+1}"
        ax.set_ylabel(lbl,fontsize=7,rotation=0,labelpad=75,va='center')
        ax.axis('off')
        axes[row,1].imshow(ov(gt,[30,120,255])); axes[row,1].axis('off')
        m=s['m']
        axes[row,2].imshow(ov(pm_mask,[220,60,60]))
        axes[row,2].set_title(f"Dice={m['dice']:.3f}  IoU={m['iou']:.3f}  CBL={m['cbl']:.3f}",
                              fontsize=8,color='darkred',pad=2); axes[row,2].axis('off')
        axes[row,3].imshow(diff)
        axes[row,3].set_title(f"Pre={m['precision']:.3f}  Rec={m['recall']:.3f}  HD95={m['hd95']:.1f}px",
                              fontsize=8,color='saddlebrown',pad=2); axes[row,3].axis('off')
    plt.tight_layout()
    safe=title.replace(' ','_').replace('|','').replace('/','_').replace('(','').replace(')','')[:40].lower()
    plt.savefig(f'{safe}.png',dpi=100,bbox_inches='tight'); plt.show()
    print(f'  ✅ {safe}.png  ({nr} hàng)')

pga_vis_all(easy_pga_vis, f'PGA | EASY — {N_SHOW} stems / {len(easy_pga_vis)} poly (metrics N={N_METRIC})')
pga_vis_all(hard_pga_vis,  f'PGA | HARD — {N_SHOW} stems / {len(hard_pga_vis)} poly (metrics N={N_METRIC})')


In [ ]:
# ══════════════════════════════════════════════════════
# CELL TỔNG HỢP — So sánh 3 model trên cùng 30 ảnh
# UNet/AttUNet: N=15 each | PGA: N_poly ≥ 15 (multi-component)
# ══════════════════════════════════════════════════════
import csv, os

N_EASY=len(easy_stems); N_HARD=len(hard_stems)          # =15
N_PGA_E=SEC['pga']['n_easy']; N_PGA_H=SEC['pga']['n_hard']  # ≥15

bar='═'*90
print(f'\n{bar}')
print(f'  BẢNG TỔNG KẾT  |  {N_EASY} Easy + {N_HARD} Hard ảnh  '
      f'(PGA: {N_PGA_E} + {N_PGA_H} poly)')
print(f'{bar}')
print(f"  {'Model':<14} {'Nhóm':<8} {'N':>6} {'Dice':>7} {'IoU':>7} {'Pre':>7} {'Rec':>7} {'HD95':>8} {'CBL':>7}")
print(f"  {'─'*74}")

csv_rows=[]
for model,key in [('U-Net','unet'),('AttUNet','attunet'),('PGA-UNet','pga')]:
    for grp in ['easy','hard']:
        m=SEC[key][grp]
        if key=='pga':
            n=N_PGA_E if grp=='easy' else N_PGA_H
            n_str=f'{n} poly'
        else:
            n=N_EASY if grp=='easy' else N_HARD
            n_str=f'{n} img'
        print(f"  {model:<14} {grp.upper():<8} {n_str:>6}"
              f" {m['dice']:>7.4f} {m['iou']:>7.4f} {m['precision']:>7.4f}"
              f" {m['recall']:>7.4f} {m['hd95']:>8.2f} {m['cbl']:>7.4f}")
        csv_rows.append([model,grp.upper(),n]+[f"{m[k]:.4f}" for k in KEYS])
    print(f"  {'─'*74}")

print(f"  {'Δ PGA−UNet':<14} {'EASY':<8} {'':>6}"
      f" {SEC['pga']['easy']['dice']-SEC['unet']['easy']['dice']:>+7.4f}")
print(f"  {'':14} {'HARD':<8} {'':>6}"
      f" {SEC['pga']['hard']['dice']-SEC['unet']['hard']['dice']:>+7.4f}")
print(bar)

os.makedirs('results',exist_ok=True)
with open('results/subcat_pga_vs_baseline.csv','w',newline='') as f:
    csv.writer(f).writerows([['model','group','N']+KEYS]+csv_rows)
print('→ CSV: results/subcat_pga_vs_baseline.csv')

fig,ax=plt.subplots(figsize=(8,5))
x=np.arange(2); w=0.25
for i,(label,key,clr) in enumerate([('U-Net','unet','#6699cc'),
                                      ('AttUNet','attunet','#66bb6a'),
                                      ('PGA-UNet','pga','#ef5350')]):
    vals=[SEC[key]['easy']['dice'],SEC[key]['hard']['dice']]
    bars=ax.bar(x+(i-1)*w,vals,w,label=label,color=clr,edgecolor='white')
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2,v+0.012,f'{v:.3f}',
                ha='center',fontsize=9,fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'EASY  ({N_EASY} img / PGA: {N_PGA_E} poly)',
                    f'HARD  ({N_HARD} img / PGA: {N_PGA_H} poly)'],fontsize=10)
ax.set_ylabel('Dice'); ax.set_ylim(0,1.15)
ax.legend(fontsize=10); ax.grid(axis='y',alpha=0.3)
ax.set_title('PGA-UNet vs Baseline — Easy vs Hard\n(30 ảnh phân nhóm theo U-Net Dice)',
             fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig('results/subcat_baseline_bar.png',dpi=130,bbox_inches='tight'); plt.show()
print('→ Plot: results/subcat_baseline_bar.png')
